In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import os
import glob

# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Folder where you saved airs.pt, whu.pt, etc.
CENTROIDS_FOLDER = "../centroids/" 

# --- 1. REPLICATE THE EXACT FEATURE EXTRACTOR ---
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # Must match the training script exactly
        weights = models.ResNet18_Weights.DEFAULT
        base = models.resnet18(weights=weights)
        self.features = nn.Sequential(*list(base.children())[:-1])
        
    def forward(self, x):
        x = self.features(x)
        return x.flatten(start_dim=1) 

class ModelSelector:
    def __init__(self, folder_path):
        self.device = DEVICE
        self.extractor = FeatureExtractor().to(self.device)
        self.extractor.eval()
        
        # EXACT same transforms as the generation script
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.centroids = self._load_centroids(folder_path)

    def _load_centroids(self, folder):
        """Robustly loads .pt files (handling both raw Tensors and Dictionaries)"""
        centroids = {}
        files = glob.glob(os.path.join(folder, "*.pt"))
        
        if not files:
            raise FileNotFoundError(f"No .pt files found in {folder}")

        print(f"Loading {len(files)} centroids from {folder}...")
        
        for path in files:
            try:
                # Load data to device
                data = torch.load(path, map_location=self.device)
                
                # Extract filename as dataset name (e.g., 'airs.pt' -> 'airs')
                name = os.path.splitext(os.path.basename(path))[0]

                # LOGIC TO HANDLE DIFFERENT SAVE FORMATS
                vec = None
                
                # Case A: Saved as a raw Tensor (most likely based on your snippet)
                if isinstance(data, torch.Tensor):
                    vec = data
                # Case B: Saved as a Dictionary (common in checkpoints)
                elif isinstance(data, dict):
                    # Try to find a tensor in the dict values
                    for k, v in data.items():
                        if isinstance(v, torch.Tensor):
                            vec = v
                            break
                
                if vec is not None:
                    # Ensure shape is [1, 512] for matrix math
                    if vec.dim() == 1:
                        vec = vec.unsqueeze(0)
                    
                    # Normalize centroid (just to be safe, even if done before)
                    vec = F.normalize(vec, p=2, dim=1)
                    centroids[name] = vec
                    print(f"  [OK] Loaded: {name}")
                else:
                    print(f"  [Error] Could not find tensor in {name}")

            except Exception as e:
                print(f"  [Fail] {path}: {e}")
                
        return centroids

    def predict_best_dataset(self, image_path):
        """
        Calculates similarity between input image and all dataset centroids.
        """
        # 1. Load Image
        try:
            img = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error opening image: {e}")
            return None

        # 2. Preprocess
        img_t = self.transform(img).unsqueeze(0).to(self.device)

        # 3. Extract Features
        with torch.no_grad():
            input_vec = self.extractor(img_t)
            # Normalize input vector (Crucial for Cosine Similarity)
            input_vec = F.normalize(input_vec, p=2, dim=1)

        # 4. Calculate Scores
        scores = {}
        for name, centroid in self.centroids.items():
            # Dot product of two normalized vectors = Cosine Similarity
            # input_vec: [1, 512] * centroid.T: [512, 1] -> [1, 1]
            score = torch.mm(input_vec, centroid.T).item()
            scores[name] = score

        # 5. Rank and Return
        if not scores:
            return None
            
        best_dataset = max(scores, key=scores.get)
        
        # Optional: Print detailed scores
        # print(f"\n--- Similarity Scores for {os.path.basename(image_path)} ---")
        # for k, v in sorted(scores.items(), key=lambda item: item[1], reverse=True):
        #     print(f"{k}: {v:.4f}")
            
        return best_dataset

In [10]:
# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # 1. Initialize
    selector = ModelSelector(CENTROIDS_FOLDER)
    
    # 2. Provide an image
    test_img = "../dataset/inria-aerial-images-sample/images/austin14.tif"
    
    if os.path.exists(test_img):
        best_model_name = selector.predict_best_dataset(test_img)
        print(f"\n🏆 Best matching model: {best_model_name}")
        
        # 3. Example Logic to load that specific model
        # model_path = f"weights/{best_model_name}_best.pth.tar"
        # print(f"Loading weights from: {model_path}")
        # model.load_state_dict(torch.load(model_path))

Loading 5 centroids from ../centroids/...
  [OK] Loaded: airs_centroid
  [OK] Loaded: ghandinagar_centroid
  [OK] Loaded: inria_centroid
  [OK] Loaded: nacala_centroid
  [OK] Loaded: whu_centroid

🏆 Best matching model: inria_centroid
